# LaBSE Indic–Indic Alignment Benchmark: FLORES-200 + Samanantar

This notebook evaluates **only LaBSE** for **Indic–Indic sentence alignment**.

It runs two benchmark tracks:

1. **FLORES-200 Indic–Indic**: clean multi-way aligned benchmark. Row `i` has the same meaning across all languages.
2. **Samanantar Indic–Indic**: Samanantar is available as English–Indic on Hugging Face, so this notebook builds Indic–Indic pairs using English as a pivot, unless you provide direct Indic–Indic Samanantar files in Drive.

Main metrics:

- `mean_gold_cosine`: alignment of true translation pairs
- `pct_gt_0_80`: threshold-style score similar to Nikunj's benchmark
- `accuracy_at_1`: retrieval success; correct translation must be nearest neighbour
- `recall_at_10`, `mrr`
- `cosine_gap`: gold-pair cosine minus random wrong-pair cosine

**Recommended first run:** keep `MAX_SAMANANTAR_PAIRS_PER_PAIR = 1000` and `MAX_FLORES_EXAMPLES = 0`.

In [ ]:
from pathlib import Path
import sys

for _candidate in (Path.cwd(), *Path.cwd().parents):
    _guard_dir = _candidate / "scripts"
    if (_guard_dir / "import_guard.py").exists():
        if str(_guard_dir) not in sys.path:
            sys.path.insert(0, str(_guard_dir))
        break
else:
    raise RuntimeError("Could not locate scripts/import_guard.py. Run this notebook from the WSAI workspace or copy the guard module alongside it.")

from import_guard import install_pandas_guards
install_pandas_guards()


In [ ]:
# Colab setup
!pip -q uninstall -y torchvision torchaudio torchtext fastai timm -q
!pip -q install   "numpy==2.0.2"   "scipy==1.15.3"   "scikit-learn==1.6.1"   "transformers==4.48.3"   "sentence-transformers==3.4.1"   "datasets==3.2.0"   "accelerate==1.3.0"   "pandas==2.2.2"   "tqdm==4.67.1"   "matplotlib==3.10.0"   "pyyaml==6.0.2"   "sentencepiece==0.2.0"

In [ ]:
from google.colab import drive

USE_DRIVE = True

if USE_DRIVE:
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/labse_indic_indic_benchmark'
else:
    BASE_DIR = '/content/labse_indic_indic_benchmark'

print('Saving outputs to:', BASE_DIR)

In [ ]:
import gc
import os
import re
import tarfile
import shutil
import random
import urllib.request
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from datasets import load_dataset
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

# We evaluate only LaBSE, as requested.
MODEL_NAME = 'labse'
MODEL_HF_ID = 'sentence-transformers/LaBSE'

BATCH_SIZE = 32
MAX_LENGTH = 128

# FLORES settings
FLORES_SPLITS = ['dev', 'devtest']
MAX_FLORES_EXAMPLES = 0   # 0 = use full dev + devtest = 2009 examples

# Samanantar settings
MAX_SAMANANTAR_STREAM_ROWS_PER_LANG = 200000   # rows to cache per English-Indic language
MAX_SAMANANTAR_PAIRS_PER_PAIR = 1000           # final Indic-Indic pairs per direction
SHUFFLE_SAMANANTAR_STREAM = False              # keep False to improve English-pivot overlap

# If you have direct Indic-Indic Samanantar files, set this to that folder.
# Expected filenames: as-bn.csv / as_bn.csv / as-bn.tsv / as_bn.tsv etc.
# Expected columns: source_text,target_text OR src,tgt OR source,target.
DIRECT_SAMANANTAR_INDIC_DIR = None

OUTPUT_DIR = Path(BASE_DIR) / 'outputs'
for sub in ['flores', 'samanantar', 'embeddings', 'samples', 'plots', 'errors']:
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)

RUN_TAG = f'labse_indic_indic_l{MAX_LENGTH}'
print('Output dir:', OUTPUT_DIR)
print('Run tag:', RUN_TAG)

## 1. Indic language list

The Samanantar Indic–Indic table shared for this task covers 11 Indic languages. We use the same 11 languages for both FLORES-200 and Samanantar so that both benchmarks are comparable.

In [ ]:
# 11 Indic languages from the Samanantar Indic-Indic matrix
# key = Samanantar short code
# flores = FLORES-200 code
INDIC_LANGS = {
    'as': {'name': 'Assamese',  'flores': 'asm_Beng'},
    'bn': {'name': 'Bengali',   'flores': 'ben_Beng'},
    'gu': {'name': 'Gujarati',  'flores': 'guj_Gujr'},
    'hi': {'name': 'Hindi',     'flores': 'hin_Deva'},
    'kn': {'name': 'Kannada',   'flores': 'kan_Knda'},
    'ml': {'name': 'Malayalam', 'flores': 'mal_Mlym'},
    'mr': {'name': 'Marathi',   'flores': 'mar_Deva'},
    'or': {'name': 'Odia',      'flores': 'ory_Orya'},
    'pa': {'name': 'Punjabi',   'flores': 'pan_Guru'},
    'ta': {'name': 'Tamil',     'flores': 'tam_Taml'},
    'te': {'name': 'Telugu',    'flores': 'tel_Telu'},
}

LANG_CODES = list(INDIC_LANGS.keys())
DIRECTED_PAIRS = [(a, b) for a in LANG_CODES for b in LANG_CODES if a != b]

print('Languages:', len(LANG_CODES), LANG_CODES)
print('Directed Indic-Indic pairs:', len(DIRECTED_PAIRS))  # 11 * 10 = 110

## 2. Load LaBSE

LaBSE is a sentence-transformer model, so we use its built-in `encode()` method.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(MODEL_HF_ID, device=DEVICE)
if DEVICE == 'cuda':
    model = model.half()

print('Loaded:', MODEL_HF_ID)

In [ ]:
def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def encode_cached(texts: List[str], cache_path: Path, batch_size: int = BATCH_SIZE) -> np.ndarray:
    """Encode text list with LaBSE and cache to .npy."""
    cache_path.parent.mkdir(parents=True, exist_ok=True)

    if cache_path.exists():
        emb = np.load(cache_path)
        if emb.shape[0] == len(texts):
            print('Loaded cache:', cache_path.name, emb.shape)
            return emb.astype('float32')
        else:
            print('Cache size mismatch, recomputing:', cache_path.name, emb.shape, 'expected', len(texts))

    emb = model.encode(
        texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).astype('float32')

    np.save(cache_path, emb)
    print('Saved cache:', cache_path.name, emb.shape)
    return emb

## 3. Metric functions

These metrics are used for both FLORES and Samanantar.

For each source-target pair, row `i` is treated as the correct translation pair.

In [ ]:
def compute_alignment_and_retrieval_metrics(
    src_emb: np.ndarray,
    tgt_emb: np.ndarray,
    seed: int = 42,
    top_ks: Tuple[int, ...] = (1, 5, 10),
) -> Dict[str, float]:
    """Compute average cosine, threshold scores, cosine gap, and retrieval metrics."""
    sim = cosine_similarity(src_emb, tgt_emb).astype('float32')
    n = sim.shape[0]

    # Gold translation pair scores: source i with target i
    gold = np.diag(sim)

    # Random wrong-pair scores
    rng = np.random.default_rng(seed)
    neg_idx = rng.permutation(n)
    for i in range(n):
        if neg_idx[i] == i:
            neg_idx[i] = (neg_idx[i] + 1) % n
    random_scores = sim[np.arange(n), neg_idx]

    # Retrieval ranks
    sorted_idx = np.argsort(-sim, axis=1)
    ranks = np.empty(n, dtype=np.int64)
    for i in range(n):
        ranks[i] = int(np.where(sorted_idx[i] == i)[0][0]) + 1

    out = {
        'n_examples': int(n),
        'mean_gold_cosine': float(np.mean(gold)),
        'median_gold_cosine': float(np.median(gold)),
        'std_gold_cosine': float(np.std(gold)),
        'min_gold_cosine': float(np.min(gold)),
        'max_gold_cosine': float(np.max(gold)),
        'pct_gt_0_50': float(np.mean(gold > 0.50)),
        'pct_gt_0_60': float(np.mean(gold > 0.60)),
        'pct_gt_0_70': float(np.mean(gold > 0.70)),
        'pct_gt_0_80': float(np.mean(gold > 0.80)),
        'random_cosine_mean': float(np.mean(random_scores)),
        'random_cosine_std': float(np.std(random_scores)),
        'cosine_gap': float(np.mean(gold) - np.mean(random_scores)),
        'mrr': float(np.mean(1.0 / ranks)),
        'mean_rank': float(np.mean(ranks)),
        'median_rank': float(np.median(ranks)),
    }

    for k in top_ks:
        out[f'recall_at_{k}'] = float(np.mean(ranks <= k))

    out['accuracy_at_1'] = out['recall_at_1']
    return out


def collect_errors(src_texts, tgt_texts, src_emb, tgt_emb, n_examples=20):
    sim = cosine_similarity(src_emb, tgt_emb).astype('float32')
    pred_idx = np.argmax(sim, axis=1)

    rows = []
    for i, p in enumerate(pred_idx):
        if p != i:
            rows.append({
                'row_id': i,
                'source_text': src_texts[i],
                'gold_translation': tgt_texts[i],
                'predicted_neighbor': tgt_texts[p],
                'gold_cosine': float(sim[i, i]),
                'predicted_cosine': float(sim[i, p]),
                'margin_pred_minus_gold': float(sim[i, p] - sim[i, i]),
            })

    df = pd.DataFrame(rows)
    if df.empty:
        return df
    return df.sort_values('margin_pred_minus_gold', ascending=False).head(n_examples)

# Part A: FLORES-200 Indic–Indic benchmark

FLORES-200 is the cleanest benchmark here because all languages are multi-way aligned by row index.

In [ ]:
FLORES_URL = 'https://dl.fbaipublicfiles.com/nllb/flores200_dataset.tar.gz'
FLORES_CACHE_DIR = Path('/content/flores200_cache')


def download_file(url: str, output_path: Path):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req) as response, open(output_path, 'wb') as out_file:
        shutil.copyfileobj(response, out_file)


def ensure_flores200_downloaded(cache_dir: Path = FLORES_CACHE_DIR) -> Path:
    cache_dir.mkdir(parents=True, exist_ok=True)
    dataset_dir = cache_dir / 'flores200_dataset'
    tar_path = cache_dir / 'flores200_dataset.tar.gz'

    if dataset_dir.exists():
        return dataset_dir

    if not tar_path.exists():
        print('Downloading FLORES-200...')
        download_file(FLORES_URL, tar_path)

    if not tarfile.is_tarfile(tar_path):
        raise RuntimeError(f'Downloaded file is not a tar archive: {tar_path}')

    print('Extracting FLORES-200...')
    with tarfile.open(tar_path, 'r:*') as tar:
        tar.extractall(cache_dir)

    if not dataset_dir.exists():
        candidates = list(cache_dir.rglob('flores200_dataset'))
        if not candidates:
            raise FileNotFoundError('Could not find flores200_dataset folder after extraction.')
        dataset_dir = candidates[0]

    return dataset_dir


def load_flores_indic_texts() -> Dict[str, List[str]]:
    flores_dir = ensure_flores200_downloaded()
    texts_by_lang = {lang: [] for lang in LANG_CODES}

    for split in FLORES_SPLITS:
        print('
Loading FLORES split:', split)
        for lang, info in INDIC_LANGS.items():
            flores_code = info['flores']
            file_path = flores_dir / split / f'{flores_code}.{split}'
            if not file_path.exists():
                raise FileNotFoundError(f'Missing FLORES file: {file_path}')
            with open(file_path, 'r', encoding='utf-8') as f:
                lines = [line.strip() for line in f if line.strip()]
            texts_by_lang[lang].extend(lines)
            print(f'{lang}: +{len(lines)}')

    if MAX_FLORES_EXAMPLES and MAX_FLORES_EXAMPLES > 0:
        for lang in texts_by_lang:
            texts_by_lang[lang] = texts_by_lang[lang][:MAX_FLORES_EXAMPLES]

    lengths = {lang: len(texts) for lang, texts in texts_by_lang.items()}
    print('
FLORES lengths:', lengths)
    if len(set(lengths.values())) != 1:
        raise ValueError(f'FLORES lengths are not equal: {lengths}')

    return texts_by_lang

flores_texts_by_lang = load_flores_indic_texts()
print('
Example:')
print('hi:', flores_texts_by_lang['hi'][0])
print('ta:', flores_texts_by_lang['ta'][0])

In [ ]:
# Encode each FLORES language once
flores_emb_by_lang = {}
flores_n_tag = MAX_FLORES_EXAMPLES if MAX_FLORES_EXAMPLES and MAX_FLORES_EXAMPLES > 0 else 'full'
flores_split_tag = '_'.join(FLORES_SPLITS)

for lang, texts in flores_texts_by_lang.items():
    cache_path = OUTPUT_DIR / 'embeddings' / f'{MODEL_NAME}_FLORES_{lang}_{flores_split_tag}_n{flores_n_tag}_l{MAX_LENGTH}.npy'
    flores_emb_by_lang[lang] = encode_cached(texts, cache_path)

clear_memory()

In [ ]:
# Run FLORES Indic-Indic all-directed-pair evaluation
flores_rows = []

for src_lang, tgt_lang in tqdm(DIRECTED_PAIRS, desc='FLORES Indic-Indic pairs'):
    src_texts = flores_texts_by_lang[src_lang]
    tgt_texts = flores_texts_by_lang[tgt_lang]
    src_emb = flores_emb_by_lang[src_lang]
    tgt_emb = flores_emb_by_lang[tgt_lang]

    metrics = compute_alignment_and_retrieval_metrics(src_emb, tgt_emb, seed=SEED)

    row = {
        'dataset': 'FLORES-200',
        'model': MODEL_NAME,
        'source_language': src_lang,
        'target_language': tgt_lang,
        'language_pair': f'{src_lang}-{tgt_lang}',
        'split': flores_split_tag,
        **metrics,
    }
    flores_rows.append(row)

    errors = collect_errors(src_texts, tgt_texts, src_emb, tgt_emb, n_examples=20)
    err_path = OUTPUT_DIR / 'errors' / f'{MODEL_NAME}_FLORES_{src_lang}-{tgt_lang}_{flores_split_tag}_n{flores_n_tag}_errors.csv'
    errors.to_csv(err_path, index=False)

flores_metrics_df = pd.DataFrame(flores_rows)
flores_metrics_path = OUTPUT_DIR / 'flores' / f'{MODEL_NAME}_FLORES_INDIC_INDIC_{flores_split_tag}_n{flores_n_tag}.csv'
flores_metrics_df.to_csv(flores_metrics_path, index=False)

print('Saved FLORES metrics:', flores_metrics_path)
print('Shape:', flores_metrics_df.shape)
display(flores_metrics_df.head())

# Part B: Samanantar Indic–Indic benchmark

The public Hugging Face Samanantar dataset is English–Indic. For Indic–Indic evaluation, this notebook does one of two things:

1. **Direct file mode**: if you provide pre-built Indic–Indic Samanantar files in Drive, it loads them.
2. **English-pivot mode**: otherwise, it builds Indic–Indic pairs by joining English–Indic Samanantar subsets on the same English sentence.

This follows the same idea as the Samanantar paper's Indic–Indic construction: using English as a pivot.

In [ ]:
def normalize_for_join(text: str) -> str:
    text = str(text).strip().lower()
    text = re.sub(r'\s+', ' ', text)
    return text


def clean_text(x) -> str:
    if x is None:
        return ''
    return str(x).strip()


def extract_en_indic_text(row: dict, config_code: str) -> Tuple[str, str]:
    # Most common Samanantar style
    if 'src' in row and 'tgt' in row:
        return clean_text(row['src']), clean_text(row['tgt'])
    if 'source' in row and 'target' in row:
        return clean_text(row['source']), clean_text(row['target'])
    if 'en' in row and config_code in row:
        return clean_text(row['en']), clean_text(row[config_code])
    if 'eng' in row and config_code in row:
        return clean_text(row['eng']), clean_text(row[config_code])
    raise KeyError(f'Could not find text fields. Available keys: {list(row.keys())}')


def load_samanantar_en_indic_cache(lang: str) -> pd.DataFrame:
    """Load English-Indic Samanantar sample for one language and cache it."""
    cache_path = OUTPUT_DIR / 'samples' / f'samanantar_EN_{lang}_stream{MAX_SAMANANTAR_STREAM_ROWS_PER_LANG}_shuffle{SHUFFLE_SAMANANTAR_STREAM}.csv'
    if cache_path.exists():
        df = pd.read_csv(cache_path)
        print('Loaded cached EN-Indic sample:', cache_path.name, df.shape)
        return df

    print(f'Streaming ai4bharat/samanantar config={lang}')
    ds = load_dataset('ai4bharat/samanantar', lang, split='train', streaming=True)
    if SHUFFLE_SAMANANTAR_STREAM:
        ds = ds.shuffle(seed=SEED, buffer_size=10_000)

    rows = []
    seen = set()
    for row in tqdm(ds, total=MAX_SAMANANTAR_STREAM_ROWS_PER_LANG, desc=f'Sampling EN-{lang}'):
        en_text, indic_text = extract_en_indic_text(row, lang)
        if len(en_text) < 3 or len(indic_text) < 3:
            continue
        if len(en_text) > 500 or len(indic_text) > 500:
            continue

        en_norm = normalize_for_join(en_text)
        if not en_norm or en_norm in seen:
            continue
        seen.add(en_norm)

        rows.append({
            'en_norm': en_norm,
            'english_text': en_text,
            f'{lang}_text': indic_text,
        })

        if len(rows) >= MAX_SAMANANTAR_STREAM_ROWS_PER_LANG:
            break

    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError(f'No Samanantar rows collected for {lang}')

    df.to_csv(cache_path, index=False)
    print('Saved EN-Indic sample:', cache_path.name, df.shape)
    return df


# Load/cache all English-Indic Samanantar samples
samanantar_en_indic_by_lang = {}
for lang in LANG_CODES:
    samanantar_en_indic_by_lang[lang] = load_samanantar_en_indic_cache(lang)

print('
Samanantar EN-Indic sample sizes:')
for lang, df in samanantar_en_indic_by_lang.items():
    print(lang, df.shape)

In [ ]:
def try_load_direct_indic_pair(src_lang: str, tgt_lang: str) -> Optional[pd.DataFrame]:
    """Try loading direct Indic-Indic pair file if user has provided one."""
    if DIRECT_SAMANANTAR_INDIC_DIR is None:
        return None

    base = Path(DIRECT_SAMANANTAR_INDIC_DIR)
    if not base.exists():
        return None

    candidates = []
    for sep in ['-', '_']:
        for ext in ['csv', 'tsv', 'txt']:
            candidates.append(base / f'{src_lang}{sep}{tgt_lang}.{ext}')
            candidates.append(base / f'{tgt_lang}{sep}{src_lang}.{ext}')

    for path in candidates:
        if not path.exists():
            continue

        print('Loading direct Indic-Indic file:', path)
        sep = '	' if path.suffix in ['.tsv', '.txt'] else ','
        df = pd.read_csv(path, sep=sep)

        if {'source_text', 'target_text'}.issubset(df.columns):
            out = df[['source_text', 'target_text']].copy()
        elif {'src', 'tgt'}.issubset(df.columns):
            out = df[['src', 'tgt']].rename(columns={'src': 'source_text', 'tgt': 'target_text'})
        elif {'source', 'target'}.issubset(df.columns):
            out = df[['source', 'target']].rename(columns={'source': 'source_text', 'target': 'target_text'})
        else:
            # fallback: first two columns
            out = df.iloc[:, :2].copy()
            out.columns = ['source_text', 'target_text']

        out['source_text'] = out['source_text'].astype(str).str.strip()
        out['target_text'] = out['target_text'].astype(str).str.strip()
        out = out[(out['source_text'] != '') & (out['target_text'] != '')]

        if len(out) > MAX_SAMANANTAR_PAIRS_PER_PAIR:
            out = out.sample(MAX_SAMANANTAR_PAIRS_PER_PAIR, random_state=SEED).reset_index(drop=True)
        else:
            out = out.reset_index(drop=True)
        return out

    return None


def build_samanantar_pivot_pair(src_lang: str, tgt_lang: str) -> pd.DataFrame:
    """Build Indic-Indic pairs by joining on the same English sentence."""
    pair = f'{src_lang}-{tgt_lang}'
    cache_path = OUTPUT_DIR / 'samples' / f'samanantar_PIVOT_{pair}_n{MAX_SAMANANTAR_PAIRS_PER_PAIR}_stream{MAX_SAMANANTAR_STREAM_ROWS_PER_LANG}.csv'

    if cache_path.exists():
        df = pd.read_csv(cache_path)
        print('Loaded cached pivot pair:', cache_path.name, df.shape)
        return df

    # First try direct file mode
    direct_df = try_load_direct_indic_pair(src_lang, tgt_lang)
    if direct_df is not None and not direct_df.empty:
        direct_df['language_pair'] = pair
        direct_df['source_language'] = src_lang
        direct_df['target_language'] = tgt_lang
        direct_df.to_csv(cache_path, index=False)
        print('Saved direct pair cache:', cache_path.name, direct_df.shape)
        return direct_df

    # Otherwise use English-pivot join
    src_df = samanantar_en_indic_by_lang[src_lang][['en_norm', f'{src_lang}_text']].copy()
    tgt_df = samanantar_en_indic_by_lang[tgt_lang][['en_norm', f'{tgt_lang}_text']].copy()

    joined = src_df.merge(tgt_df, on='en_norm', how='inner')
    print(f'Pivot join {pair}: {len(joined)} overlaps')

    if len(joined) == 0:
        raise ValueError(
            f'No English-pivot overlap for {pair}. Increase MAX_SAMANANTAR_STREAM_ROWS_PER_LANG or provide direct files.'
        )

    if len(joined) > MAX_SAMANANTAR_PAIRS_PER_PAIR:
        joined = joined.sample(MAX_SAMANANTAR_PAIRS_PER_PAIR, random_state=SEED).reset_index(drop=True)
    else:
        joined = joined.reset_index(drop=True)

    out = pd.DataFrame({
        'language_pair': pair,
        'source_language': src_lang,
        'target_language': tgt_lang,
        'source_text': joined[f'{src_lang}_text'].astype(str),
        'target_text': joined[f'{tgt_lang}_text'].astype(str),
        'pivot_english_norm': joined['en_norm'].astype(str),
    })

    out.to_csv(cache_path, index=False)
    print('Saved pivot pair cache:', cache_path.name, out.shape)
    return out

In [ ]:
# Build/load Samanantar Indic-Indic pairs
samanantar_pairs_by_direction = {}
low_overlap_pairs = []

for src_lang, tgt_lang in tqdm(DIRECTED_PAIRS, desc='Building Samanantar Indic-Indic pairs'):
    pair = f'{src_lang}-{tgt_lang}'
    try:
        df = build_samanantar_pivot_pair(src_lang, tgt_lang)
        samanantar_pairs_by_direction[pair] = df
        if len(df) < MAX_SAMANANTAR_PAIRS_PER_PAIR:
            low_overlap_pairs.append((pair, len(df)))
    except Exception as e:
        print('FAILED building pair:', pair, type(e).__name__, e)

print('
Built Samanantar directions:', len(samanantar_pairs_by_direction))
if low_overlap_pairs:
    print('
Pairs with fewer than target examples:')
    print(low_overlap_pairs[:30])

# preview
first_pair = next(iter(samanantar_pairs_by_direction))
print('Preview pair:', first_pair)
display(samanantar_pairs_by_direction[first_pair].head())

In [ ]:
# Evaluate LaBSE on Samanantar Indic-Indic pairs
samanantar_rows = []

for pair, df in tqdm(samanantar_pairs_by_direction.items(), desc='Samanantar Indic-Indic evaluation'):
    src_lang = df['source_language'].iloc[0]
    tgt_lang = df['target_language'].iloc[0]
    src_texts = df['source_text'].astype(str).tolist()
    tgt_texts = df['target_text'].astype(str).tolist()

    pair_n_tag = len(df)
    src_cache = OUTPUT_DIR / 'embeddings' / f'{MODEL_NAME}_SAMANANTAR_{pair}_SRC_n{pair_n_tag}_l{MAX_LENGTH}.npy'
    tgt_cache = OUTPUT_DIR / 'embeddings' / f'{MODEL_NAME}_SAMANANTAR_{pair}_TGT_n{pair_n_tag}_l{MAX_LENGTH}.npy'

    src_emb = encode_cached(src_texts, src_cache)
    tgt_emb = encode_cached(tgt_texts, tgt_cache)

    metrics = compute_alignment_and_retrieval_metrics(src_emb, tgt_emb, seed=SEED)

    row = {
        'dataset': 'Samanantar-EnglishPivot',
        'model': MODEL_NAME,
        'source_language': src_lang,
        'target_language': tgt_lang,
        'language_pair': pair,
        **metrics,
    }
    samanantar_rows.append(row)

    errors = collect_errors(src_texts, tgt_texts, src_emb, tgt_emb, n_examples=20)
    err_path = OUTPUT_DIR / 'errors' / f'{MODEL_NAME}_SAMANANTAR_{pair}_n{pair_n_tag}_errors.csv'
    errors.to_csv(err_path, index=False)

    clear_memory()

samanantar_metrics_df = pd.DataFrame(samanantar_rows)
samanantar_metrics_path = OUTPUT_DIR / 'samanantar' / f'{MODEL_NAME}_SAMANANTAR_INDIC_INDIC_pivot_n{MAX_SAMANANTAR_PAIRS_PER_PAIR}.csv'
samanantar_metrics_df.to_csv(samanantar_metrics_path, index=False)

print('Saved Samanantar metrics:', samanantar_metrics_path)
print('Shape:', samanantar_metrics_df.shape)
display(samanantar_metrics_df.head())

# Part C: Compare FLORES-200 vs Samanantar Indic–Indic

In [ ]:
combined_df = pd.concat([flores_metrics_df, samanantar_metrics_df], ignore_index=True)
combined_path = OUTPUT_DIR / f'{MODEL_NAME}_COMBINED_FLORES_SAMANANTAR_INDIC_INDIC.csv'
combined_df.to_csv(combined_path, index=False)
print('Saved combined metrics:', combined_path)
print('Combined shape:', combined_df.shape)
display(combined_df.head())

In [ ]:
summary = (
    combined_df
    .groupby('dataset')[[
        'mean_gold_cosine', 'pct_gt_0_80', 'accuracy_at_1',
        'recall_at_10', 'mrr', 'cosine_gap', 'random_cosine_mean'
    ]]
    .mean()
    .sort_values('accuracy_at_1', ascending=False)
)

summary_path = OUTPUT_DIR / f'{MODEL_NAME}_DATASET_SUMMARY_INDIC_INDIC.csv'
summary.to_csv(summary_path)
print('Saved summary:', summary_path)
display(summary)

In [ ]:
# Pair-level summary sorted by Accuracy@1
pair_summary = (
    combined_df
    .sort_values(['dataset', 'accuracy_at_1'], ascending=[True, False])
    [[
        'dataset', 'language_pair', 'n_examples', 'mean_gold_cosine', 'pct_gt_0_80',
        'accuracy_at_1', 'recall_at_10', 'mrr', 'cosine_gap', 'random_cosine_mean'
    ]]
)

pair_summary_path = OUTPUT_DIR / f'{MODEL_NAME}_PAIR_SUMMARY_INDIC_INDIC.csv'
pair_summary.to_csv(pair_summary_path, index=False)
print('Saved pair summary:', pair_summary_path)
display(pair_summary.head(20))

In [ ]:
def plot_heatmap(df: pd.DataFrame, dataset_name: str, metric: str):
    sub = df[df['dataset'] == dataset_name].copy()
    heat = sub.pivot(index='source_language', columns='target_language', values=metric)
    heat = heat.reindex(index=LANG_CODES, columns=LANG_CODES)

    plt.figure(figsize=(9, 7))
    plt.imshow(heat, aspect='auto')
    plt.colorbar(label=metric)
    plt.xticks(range(len(heat.columns)), heat.columns)
    plt.yticks(range(len(heat.index)), heat.index)
    plt.xlabel('Target language')
    plt.ylabel('Source language')
    plt.title(f'{MODEL_NAME}: {dataset_name} Indic-Indic {metric}')

    for i in range(len(heat.index)):
        for j in range(len(heat.columns)):
            val = heat.iloc[i, j]
            if not pd.isna(val):
                plt.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=7)

    plt.tight_layout()
    plot_path = OUTPUT_DIR / 'plots' / f'{MODEL_NAME}_{dataset_name}_{metric}_heatmap.png'
    plt.savefig(plot_path, dpi=200)
    plt.show()
    print('Saved:', plot_path)

for dataset_name in combined_df['dataset'].unique():
    plot_heatmap(combined_df, dataset_name, 'accuracy_at_1')
    plot_heatmap(combined_df, dataset_name, 'mean_gold_cosine')
    plot_heatmap(combined_df, dataset_name, 'cosine_gap')

In [ ]:
# Compact bar comparison between datasets
compare_metrics = ['mean_gold_cosine', 'pct_gt_0_80', 'accuracy_at_1', 'recall_at_10', 'mrr', 'cosine_gap']
ax = summary[compare_metrics].plot(kind='bar', figsize=(12, 5))
ax.set_title('LaBSE Indic-Indic benchmark: FLORES-200 vs Samanantar')
ax.set_ylabel('Average score')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plot_path = OUTPUT_DIR / 'plots' / f'{MODEL_NAME}_FLORES_vs_SAMANANTAR_summary.png'
plt.savefig(plot_path, dpi=200)
plt.show()
print('Saved:', plot_path)

## What to report to the mentor

Use the following result interpretation:

- **FLORES-200** is the clean multi-way Indic–Indic benchmark because all languages are aligned by row.
- **Samanantar Indic–Indic** is built through English pivot unless direct Indic–Indic files are provided.
- Report both **alignment** and **uniformity/separation**:
  - Alignment: `mean_gold_cosine`, `% > 0.80`
  - Uniformity/separation: `cosine_gap`, `Accuracy@1`, `Recall@10`, `MRR`
- If a model has high mean cosine but low Accuracy@1, it means correct pairs are close but wrong candidates may also be close.